# VISTIQ Clothing Classifier — Training Notebook

Trains a MobileNetV3-Small clothing-category classifier for VISTIQ using
transfer learning, on **[clothing-dataset-small](https://github.com/alexeygrigorev/clothing-dataset-small)**
(real photographed garments, CC0 license, pre-split by its author).

This notebook must be run in **Google Colab** with a GPU runtime — the
VISTIQ development laptop has no NVIDIA GPU. It also runs on CPU (just much
slower), so it will not fail outright on a CPU-only runtime.

Mirrors the architecture/config in `ml/training/*.py` and
`ml/labels/clothing_categories.json` in the VISTIQ repo — kept consistent by
hand since a fresh Colab VM can't see local project files. If you change the
label mapping or hyperparameters here, update those files too.

**Output of this notebook:** `vistiq_clothing_classifier.pt`, to be placed at
`VistiQ/models/vistiq_clothing_classifier.pt`.

## 1. Environment setup

In [ ]:
# Colab ships torch/torchvision/scikit-learn/matplotlib/Pillow already —
# this just confirms versions and installs anything missing (idempotent).
import subprocess, sys

def ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg in ["torch", "torchvision", "sklearn", "matplotlib", "PIL"]:
    ensure(pkg if pkg != "sklearn" else "scikit-learn")

import torch, torchvision, sklearn, matplotlib, PIL
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)
print("Pillow:", PIL.__version__)

## 2. GPU detection

Prints whatever device is actually available and uses it — the notebook
does **not** assume a GPU exists, and will fall back to CPU rather than
fail.

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — training will run on CPU (much slower). "
          "In Colab: Runtime -> Change runtime type -> GPU.")
print("Using device:", device)

## 3. Dataset acquisition

`clothing-dataset-small` — 10 real-world clothing classes, pre-split into
`train/`, `validation/`, `test/`. Plain `git clone`, no Kaggle account/API
key needed. Verified against the live repository listing before this
notebook was written (not assumed from memory).

In [ ]:
import os

DATASET_DIR = "clothing-dataset-small"

if not os.path.isdir(DATASET_DIR):
    !git clone https://github.com/alexeygrigorev/clothing-dataset-small.git

print("Dataset splits:", sorted(os.listdir(DATASET_DIR)))

## 4. Dataset inspection

Real counts computed from the actual downloaded files — never hard-coded.
If the upstream repo changes, these numbers will change too, which is why
we compute rather than assume them.

In [ ]:
from pathlib import Path

DATASET_ROOT = Path(DATASET_DIR)
SPLITS = ["train", "validation", "test"]

raw_counts = {}
for split in SPLITS:
    split_dir = DATASET_ROOT / split
    raw_counts[split] = {
        class_dir.name: len(list(class_dir.glob("*")))
        for class_dir in sorted(split_dir.iterdir()) if class_dir.is_dir()
    }

for split in SPLITS:
    total = sum(raw_counts[split].values())
    print(f"--- {split} ({total} images) ---")
    for cls, n in sorted(raw_counts[split].items()):
        print(f"  {cls:<12} {n}")
    print()

In [ ]:
# A few sample images, so we can visually sanity-check the data before
# training anything on it.
import matplotlib.pyplot as plt
from PIL import Image
import random

sample_classes = random.sample(sorted(raw_counts["train"].keys()), k=5)
fig, axes = plt.subplots(1, len(sample_classes), figsize=(15, 3))
for ax, cls in zip(axes, sample_classes):
    class_dir = DATASET_ROOT / "train" / cls
    sample_path = next(class_dir.glob("*"))
    ax.imshow(Image.open(sample_path).convert("RGB"))
    ax.set_title(cls)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Label mapping

The dataset's 10 folder names, mapped onto VISTIQ's 6 wardrobe categories.
This mirrors `ml/labels/clothing_categories.json` in the VISTIQ repo — the
single documented source of truth for this mapping.

**Documented limitation:** `Outerwear` and `Accessories` each come from only
one narrow source class (`outwear`, `hat`). The trained model's notion of
"Accessories" will really only mean "hat-like" — it won't recognize bags,
jewelry, belts, etc. This is a real dataset limitation, shown honestly in
the per-class evaluation metrics later, not hidden.

In [ ]:
VISTIQ_CATEGORIES = ["Tops", "Bottoms", "Outerwear", "Dresses", "Footwear", "Accessories"]

LABEL_MAPPING = {
    "t-shirt": "Tops",
    "shirt": "Tops",
    "longsleeve": "Tops",
    "pants": "Bottoms",
    "shorts": "Bottoms",
    "skirt": "Bottoms",
    "outwear": "Outerwear",
    "dress": "Dresses",
    "shoes": "Footwear",
    "hat": "Accessories",
}

# Sanity check: every folder we actually found on disk must be mapped.
unmapped = set(raw_counts["train"].keys()) - set(LABEL_MAPPING.keys())
assert not unmapped, f"Unmapped dataset folders found: {unmapped} — update LABEL_MAPPING."
print("All dataset folders are mapped to a VISTIQ category.")

# Real VISTIQ-category counts per split (computed, not assumed).
vistiq_counts = {split: {c: 0 for c in VISTIQ_CATEGORIES} for split in SPLITS}
for split in SPLITS:
    for raw_label, n in raw_counts[split].items():
        vistiq_counts[split][LABEL_MAPPING[raw_label]] += n

for split in SPLITS:
    print(f"--- {split}: VISTIQ category counts ---")
    for cat in VISTIQ_CATEGORIES:
        print(f"  {cat:<12} {vistiq_counts[split][cat]}")
    print()

## 6. Train / validation / test split

We use the dataset author's own predefined `train/validation/test` folders
rather than re-splitting — it's a fixed, reproducible split that anyone
cloning the same repo can reference, and re-splitting on top of an
already-curated split would just add unnecessary randomness.

In [ ]:
for split in SPLITS:
    total = sum(vistiq_counts[split].values())
    print(f"{split}: {total} images")

## 7. Image preprocessing

MobileNetV3's ImageNet-pretrained weights expect 224×224 RGB input,
normalized with ImageNet's mean/std. Using anything else would feed the
pretrained backbone out-of-distribution inputs and hurt transfer learning.

In [ ]:
from torchvision import transforms

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## 8. Data augmentation

Deliberately mild. Category, color, and silhouette are the signal we want
the model to learn — so no vertical flips (clothing would look upside
down/unrealistic), no aggressive cropping, and only slight color jitter
(strong jitter would change a garment's apparent color, which actively
hurts the color-extraction step downstream).

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomRotation(degrees=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

BATCH_SIZE = 32
VISTIQ_CATEGORY_TO_IDX = {name: i for i, name in enumerate(VISTIQ_CATEGORIES)}

def remap_target_transform(raw_class_to_idx):
    raw_idx_to_vistiq_idx = {
        raw_idx: VISTIQ_CATEGORY_TO_IDX[LABEL_MAPPING[raw_label]]
        for raw_label, raw_idx in raw_class_to_idx.items()
    }
    return lambda raw_idx: raw_idx_to_vistiq_idx[raw_idx]

def load_split(split, train):
    ds = ImageFolder(
        root=str(DATASET_ROOT / split),
        transform=train_transform if train else eval_transform,
    )
    ds.target_transform = remap_target_transform(ds.class_to_idx)
    return ds

train_ds = load_split("train", train=True)
val_ds = load_split("validation", train=False)
test_ds = load_split("test", train=False)

g = torch.Generator().manual_seed(42)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"train={len(train_ds)}  validation={len(val_ds)}  test={len(test_ds)}")

## 9. Pretrained MobileNet model

MobileNetV3-Small, ImageNet-pretrained. Chosen for being lightweight
(~2.5M params), fast to fine-tune, fast enough for CPU inference on a
laptop with no GPU, and simple enough to explain end-to-end in a viva.

In [ ]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
print(model.classifier)

## 10. Classifier head

Replace the final layer (1000 ImageNet classes) with a 6-class VISTIQ head.
Keep the two layers before it — they're a general-purpose feature
projection, not ImageNet-class-specific.

In [ ]:
import torch.nn as nn

NUM_CLASSES = len(VISTIQ_CATEGORIES)
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)
model.to(device)

def freeze_backbone(m):
    for name, p in m.named_parameters():
        p.requires_grad = name.startswith("classifier")

def unfreeze_last_blocks(m, n_blocks):
    for block in list(m.features.children())[-n_blocks:]:
        for p in block.parameters():
            p.requires_grad = True

freeze_backbone(model)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters (phase 1, head only): {trainable:,} / {total:,}")

## 11. Training — Phase 1 (frozen backbone, train the head)

In [ ]:
import copy, time

def run_epoch(model, loader, criterion, optimizer, train):
    model.train(mode=train)
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total


def train_phase(model, epochs, lr, phase_name, history, best_state, patience=3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, train=True)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, train=False)
        elapsed = time.time() - start

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["phase"].append(phase_name)

        print(f"[{phase_name}] epoch {epoch}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({elapsed:.1f}s)")

        if val_acc > best_state["val_acc"]:
            best_state["val_acc"] = val_acc
            best_state["weights"] = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"[{phase_name}] no improvement for {patience} epochs — stopping early.")
                break

    return history, best_state


PHASE1_EPOCHS = 8
PHASE1_LR = 1e-3

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "phase": []}
best_state = {"val_acc": 0.0, "weights": None}

history, best_state = train_phase(
    model, PHASE1_EPOCHS, PHASE1_LR, "phase1_head", history, best_state
)

## 12. Validation (Phase 1)

Validation accuracy/loss were already tracked every epoch above (that's
what makes early stopping and best-checkpoint selection possible). This
cell just plots Phase 1 in isolation before we move on to fine-tuning.

In [ ]:
phase1_len = history["phase"].count("phase1_head")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"][:phase1_len], label="train")
ax1.plot(history["val_loss"][:phase1_len], label="validation")
ax1.set_title("Phase 1 — Loss"); ax1.set_xlabel("epoch"); ax1.legend()

ax2.plot(history["train_acc"][:phase1_len], label="train")
ax2.plot(history["val_acc"][:phase1_len], label="validation")
ax2.set_title("Phase 1 — Accuracy"); ax2.set_xlabel("epoch"); ax2.legend()
plt.tight_layout(); plt.show()

print(f"Best validation accuracy so far: {best_state['val_acc']:.4f}")

## 13. Fine-tuning — Phase 2 (unfreeze last blocks, low LR)

Unfreezes only the last few backbone blocks and trains at a much lower
learning rate, so pretrained low-level features (edges, textures) stay
intact while higher-level features adapt to clothing photos specifically.

In [ ]:
PHASE2_EPOCHS = 6
PHASE2_LR = 1e-5
PHASE2_UNFREEZE_LAST_N_BLOCKS = 3

unfreeze_last_blocks(model, PHASE2_UNFREEZE_LAST_N_BLOCKS)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters (phase 2, backbone tail + head): {trainable:,} / {total:,}")

history, best_state = train_phase(
    model, PHASE2_EPOCHS, PHASE2_LR, "phase2_finetune", history, best_state
)

# Use the best checkpoint seen across BOTH phases, not just whatever the
# final epoch happened to land on.
if best_state["weights"] is not None:
    model.load_state_dict(best_state["weights"])
print(f"Best validation accuracy overall: {best_state['val_acc']:.4f}")

## 14. Evaluation (held-out test set)

Test images were never used for training or for early-stopping decisions —
this is the first and only time the model sees them.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
    return np.array(all_labels), np.array(all_preds)

test_labels, test_preds = collect_predictions(model, test_loader)
test_accuracy = float((test_labels == test_preds).mean())

print(f"Test accuracy: {test_accuracy:.4f}  ({len(test_labels)} test images)")
print()
report = classification_report(
    test_labels, test_preds, target_names=VISTIQ_CATEGORIES, zero_division=0
)
print(report)

## 15. Confusion matrix

In [ ]:
cm = confusion_matrix(test_labels, test_preds, labels=list(range(len(VISTIQ_CATEGORIES))))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(VISTIQ_CATEGORIES))); ax.set_yticks(range(len(VISTIQ_CATEGORIES)))
ax.set_xticklabels(VISTIQ_CATEGORIES, rotation=45, ha="right")
ax.set_yticklabels(VISTIQ_CATEGORIES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("VISTIQ clothing classifier — confusion matrix (test set)")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## 16. Accuracy and loss graphs (full training run, both phases)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="train")
ax1.plot(history["val_loss"], label="validation")
phase_boundary = history["phase"].count("phase1_head")
ax1.axvline(phase_boundary - 0.5, color="gray", linestyle="--", label="phase 1 -> 2")
ax1.set_title("Loss (full run)"); ax1.set_xlabel("epoch"); ax1.legend()

ax2.plot(history["train_acc"], label="train")
ax2.plot(history["val_acc"], label="validation")
ax2.axvline(phase_boundary - 0.5, color="gray", linestyle="--", label="phase 1 -> 2")
ax2.set_title("Accuracy (full run)"); ax2.set_xlabel("epoch"); ax2.legend()
plt.tight_layout(); plt.show()

## 17. Single-image prediction

Demonstrates the full pipeline VISTIQ actually uses: trained-model category
prediction (real ML, with confidence) + deterministic dominant-color
extraction (plain image processing — mirrors `ml/inference/color.py`, kept
separate from the ML confidence on purpose).

In [ ]:
def extract_dominant_color(pil_image, palette_colors=8, bg_white_threshold=235):
    rgb = pil_image.convert("RGB")
    rgb.thumbnail((120, 120))
    pixels = list(rgb.getdata())
    fg = [p for p in pixels if not (p[0] > bg_white_threshold and p[1] > bg_white_threshold and p[2] > bg_white_threshold)]
    sample = fg if fg else pixels
    sample_img = Image.new("RGB", (len(sample), 1))
    sample_img.putdata(sample)
    quantized = sample_img.quantize(colors=palette_colors, method=Image.MEDIANCUT)
    palette = quantized.getpalette()
    dominant_index = max(quantized.getcolors(), key=lambda c: c[0])[1]
    return tuple(palette[dominant_index * 3: dominant_index * 3 + 3])

VISTIQ_COLOR_PALETTE = {
    "Black": (24, 24, 27), "White": (244, 244, 245), "Grey": (156, 163, 175),
    "Navy": (36, 59, 85), "Indigo": (60, 74, 117), "Beige": (217, 205, 184),
    "Brown": (107, 74, 52), "Green": (63, 108, 79), "Blue": (59, 108, 191),
    "Red": (178, 42, 42), "Yellow": (214, 184, 60), "Pink": (214, 145, 168),
    "Purple": (110, 72, 145), "Orange": (201, 111, 53),
}

def nearest_vistiq_color(rgb):
    return min(VISTIQ_COLOR_PALETTE, key=lambda name: sum((a - b) ** 2 for a, b in zip(rgb, VISTIQ_COLOR_PALETTE[name])))


@torch.no_grad()
def predict_single_image(pil_image):
    model.eval()
    tensor = eval_transform(pil_image.convert("RGB")).unsqueeze(0).to(device)
    probs = torch.softmax(model(tensor), dim=1)[0].cpu()
    top_idx = int(torch.argmax(probs))
    dominant_rgb = extract_dominant_color(pil_image)
    return {
        "category": VISTIQ_CATEGORIES[top_idx],
        "confidence": round(float(probs[top_idx]), 4),
        "color": nearest_vistiq_color(dominant_rgb),
    }


# Try it on a handful of real, unseen test images.
sample_test_paths = random.sample(test_ds.samples, k=min(4, len(test_ds.samples)))
fig, axes = plt.subplots(1, len(sample_test_paths), figsize=(4 * len(sample_test_paths), 4))
if len(sample_test_paths) == 1:
    axes = [axes]
for ax, (path, _raw_idx) in zip(axes, sample_test_paths):
    image = Image.open(path)
    result = predict_single_image(image)
    ax.imshow(image.convert("RGB"))
    ax.set_title(f"{result['category']} ({result['confidence']:.0%})\ncolor: {result['color']}")
    ax.axis("off")
plt.tight_layout(); plt.show()

## 18. Export trained model

Saves a plain `state_dict` (not the full pickled module) — smaller, more
portable, and requires the receiving code to build the same architecture
first (which `ml/training/model.build_model()` does), which is the standard,
safer PyTorch export pattern.

In [ ]:
EXPORT_FILENAME = "vistiq_clothing_classifier.pt"
torch.save(model.state_dict(), EXPORT_FILENAME)

import os
size_mb = os.path.getsize(EXPORT_FILENAME) / (1024 * 1024)
print(f"Saved {EXPORT_FILENAME} ({size_mb:.2f} MB)")

## 19. Download / export instructions

1. Run the cell below to download `vistiq_clothing_classifier.pt` from Colab.
2. Move the downloaded file to `VistiQ/models/vistiq_clothing_classifier.pt`
   on your machine (create the `models/` folder if it isn't there — it
   already exists in the VISTIQ repo with a README explaining this exact
   step).
3. No code changes are needed — `ml/inference/predict.py` and the FastAPI
   `/analyze-clothing` endpoint already look for the model at that exact
   path.
4. Restart the FastAPI backend so it picks up the newly-present model file.

In [ ]:
from google.colab import files
files.download(EXPORT_FILENAME)